# STAT 207 Homework 11 [25 points]

## Regularization Models for Linear Relationships

Due: Monday, May 4, end of day (11:59 pm CT)

Late submissions accepted until Tuesday, May 5 at noon

<hr>

## Imports 

Run the following code cell to import the necessary packages into the file.  You may import additional packages, as needed for this assignment.

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

## The Data

With available climate data dating back many decades and the prevalence of climate change, humans are looking to understand exactly how different features of the climate affect the temperatures globally.  For this assignment, we will look to understand how the **global temperature** fluctuates based on other environmental features.

We will use various atmospheric and temperature measures over 309 months from 1983 to 2008, with the following variables:

- **`Year`**: the observation year
- **`Month`**: the observation month, recorded with numbers 1 to 12
- **`MEI`**: Multivariate El Nino Southern Oscillation Index (MEI), measuring the affects of the El Nino weather pattern
- **`CO2`**: atmospheric concentration of carbon dioxide (in ppmv, parts per million by volume)
- **`CH4`**: atmospheric concentration of methane (in ppmv)
- **`N2O`**: atmospheric concentration of nitrous oxide (in ppmv)
- **`CFC-11`**: atmospheric concentration of CCl3F or trichlorofluoromethane (in ppbv, parts per billion by volume)
- **`CFC-12`**: atmospheric concentration of CCl2F2 or dichlorodifluoromethane (in ppbv)
- **`TSI`**: the total solar irradiance (TSI) (in W/m2), measuring the rate at which the sun's energy is deposited per unit area.
- **`Aerosols`**: the mean stratospheric aerosol optical depth at 500 nm, a measure associated with volcanic activity
- **`Temp`**: the difference in the average global temperature for that month (in Celsius) and a reference value

The ESRL/NOAA Physical Sciences Division reports the MEI; atmospheric concentrations are measured by the ESRL/NOAA Global Monitoring Division; the SOLARIS-HEPPA project website provides the TSI; the Godard Institute for Space Studies at NASA reports the Aerosols; and the Climatic Research Unit at the University of East Anglia reports the Temp.

Run the code in the cell below to read in the cleaned data for this document.  The data is saved as `df` with this code.  

In [5]:
df = pd.read_csv('climate_change.csv')
df_train = df[df['Year'] <= 2006]
df_test = df[df['Year'] >= 2007]
X_train = df_train.drop(['Year', 'Month', 'Temp'], axis = 1)
X_test = df_test.drop(['Year', 'Month', 'Temp'], axis = 1)
y_train = df_train['Temp']
y_test = df_test['Temp']

## 1. Summarize Data [1.5 points]

Above, we set aside a training data.  We didn't randomly select our training and test set; instead, imagine that we fit a model using the available data in 2006 in our training data.  We'll then use the data that we collect in the following two years as the test set to evaluate this model.

As defined in our X_train and X_test above, our response variable for this assignment with be **`Temp`**.  We'll use all variables except the **`Year`** and **`Month`** as our predictor variables.

**a)** Scale our predictor variables in the training data.

In [6]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns = X_train.columns,
    index = X_train.index
)

X_train_scaled.describe()

,MEI,CO2,CH4,N2O,CFC-11,CFC-12,TSI,Aerosols
count,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,284.000000
mean,3.752867e-17,2.501911e-15,2.301758e-15,-2.001529e-16,-1.150879e-15,2.501911e-16,-4.336813e-13,0.000000
std,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765
min,-2.077501e+00,-1.860344e+00,-2.543388e+00,-1.680002e+00,-2.919383e+00,-2.444829e+00,-1.685916e+00,-0.538056
25%,-7.165109e-01,-7.968154e-01,-6.469486e-01,-8.421301e-01,-1.398208e-01,-5.373796e-01,-8.659712e-01,-0.501341
50%,-3.601564e-02,-1.334784e-01,2.799670e-01,-1.700429e-01,3.764272e-01,4.728576e-01,-1.171738e-01,-0.384524
75%,5.992210e-01,8.030748e-01,7.851835e-01,9.414536e-01,7.141010e-01,7.932329e-01,7.435250e-01,-0.124187
max,2.865383e+00,2.063634e+00,1.366734e+00,1.851271e+00,9.072212e-01,8.414196e-01,3.032543e+00,4.394997


**b)** Now, we want to be sure that we also scale our test data, using the same scaling as applied to our training data.  Apply your scaling algorithm from **part a** to the test data.  

*Note*: This does not include re-fitting your scaling process.  You will re-use your scaling process from **part a**, simply transforming your test data with the same scaling process.

In [7]:
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns = X_test.columns,
    index = X_test.index
)

X_test_scaled.describe()


,MEI,CO2,CH4,N2O,CFC-11,CFC-12,TSI,Aerosols
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,-0.917795,2.036889,1.121218,1.984680,-0.314865,0.720071,-0.982834,-0.455588
std,0.690182,0.179041,0.250031,0.114226,0.065486,0.024218,0.079206,0.020206
min,-2.130303,1.706352,0.571373,1.826219,-0.401280,0.690306,-1.109496,-0.487991
25%,-1.493181,1.895939,1.020765,1.873744,-0.375720,0.693835,-1.064123,-0.471303
50%,-0.940110,2.023790,1.145081,2.020740,-0.301021,0.725222,-0.965827,-0.454614
75%,-0.407244,2.164996,1.265504,2.052897,-0.281332,0.732229,-0.936619,-0.441264
max,0.681118,2.371878,1.499001,2.215683,-0.196418,0.763259,-0.848371,-0.411225


**c)** I'd suggest turning to Q1 on Gradescope here.

You'll likely need to further explore the data for Gradescope Q1.  You can use this section for any exploration that you'd like, although there are no points associated with this part.

In [9]:
X_train.std().max().round(2)


np.float64(59.05)

In [10]:
round(X_train.std().min(), 2)

np.float64(0.03)

In [12]:
X_train_scaled.mean()
X_train_scaled.std()

MEI         1.001765
CO2         1.001765
CH4         1.001765
N2O         1.001765
CFC-11      1.001765
CFC-12      1.001765
TSI         1.001765
Aerosols    1.001765
dtype: float64

In [14]:
round(X_train_scaled.std().min(), 2)


np.float64(1.0)

## 2. Fitting A Model [1 point]

**a)** Fit a LASSO model with $\lambda = 0.06$ to the training data, including all variables except the year and month variables, as set up in the provided $X$ features matrix.  Print the coefficients for this model.

In [15]:
lasso_model = Lasso(alpha = 0.06)

lasso_model.fit(X_train_scaled, y_train)

pd.Series(lasso_model.coef_, index = X_train_scaled.columns)


MEI         0.000000
CO2         0.079893
CH4         0.000000
N2O         0.002758
CFC-11      0.000000
CFC-12      0.000000
TSI         0.000000
Aerosols   -0.000000
dtype: float64

**b)** I'd suggest turning to Q2 on Gradescope here.

I don't anticipate a need to perform any more calculations or analyses for this problem.  This space is available for any optional calculations or analyses that you'd like, although there are no points associated with this part.

## 3. Picking a Best Model [2.5 points]

Instead of using a LASSO model, we decide that we'd rather move forward with a **ridge regression** model.  We don't know which $\lambda$ to use for this model, so let's explore which value of $\lambda$ might be most appropriate for a ridge regression model.

To do this, we'll explore $\lambda$ values between 0.05 and 1 exploring by every 0.05.  We can do this with code using:

`for m in range(1, 21):`

`    alph = m / 20`

The following code sets up the folds for cross-validation.

In [16]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

cross_val = KFold(n_splits = 10, shuffle = True, random_state = 202605)

**a)** Use 10-fold cross-validation to explore the $R^2$ values for each of the $\lambda$s as defined above.

In [17]:
for m in range(1, 21):
    alph = m / 20

    ridge_model = Ridge(alpha = alph)
    r2_scores = cross_val_score(
        ridge_model,
        X_train_scaled,
        y_train,
        cv = cross_val,
        scoring = 'r2'
    )

    print("lambda =", alph)
    print(r2_scores)
    print("mean R^2 =", r2_scores.mean())
    print()


/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 

lambda = 0.05
[0.71695016 0.73307827 0.76040604 0.76705368 0.75535774 0.76813387
 0.7916397  0.44898877 0.69064681 0.7316102 ]
mean R^2 = 0.7163865238579991

lambda = 0.1
[0.71751345 0.73436974 0.76204491 0.76658304 0.75487552 0.7685196
 0.79112719 0.44783017 0.68993342 0.73124019]
mean R^2 = 0.7164037228590681

lambda = 0.15
[0.71801403 0.73555043 0.76351272 0.76611995 0.75440612 0.76886423
 0.79063015 0.4467188  0.68923808 0.73087899]
mean R^2 = 0.7163933508213389

lambda = 0.2
[0.71846041 0.73663341 0.76483248 0.7656656  0.75395009 0.76917307
 0.79014906 0.4456538  0.68856234 0.73052688]
mean R^2 = 0.7163607127582174

lambda = 0.25
[0.71885973 0.73762985 0.76602345 0.76522078 0.75350766 0.76945063
 0.78968412 0.44463395 0.68790713 0.73018397]
mean R^2 = 0.7163101255154943

lambda = 0.3
[0.71921802 0.73854932 0.76710191 0.764786   0.75307889 0.76970075
 0.78923529 0.44365779 0.68727292 0.72985027]
mean R^2 = 0.7162451158807255

lambda = 0.35
[0.71954044 0.73940012 0.76808162 0.764361

/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 

**b)** Print the $R^2$ values for each of the individual folds of your optimal $\lambda$.

In [18]:
best_alpha = None
best_mean_r2 = -999

for m in range(1, 21):
    alph = m / 20

    ridge_model = Ridge(alpha = alph)
    r2_scores = cross_val_score(
        ridge_model,
        X_train_scaled,
        y_train,
        cv = cross_val,
        scoring = 'r2'
    )

    if r2_scores.mean() > best_mean_r2:
        best_mean_r2 = r2_scores.mean()
        best_alpha = alph

print("Optimal lambda:", best_alpha)
print("Mean R^2:", best_mean_r2)

best_ridge_model = Ridge(alpha = best_alpha)
best_r2_scores = cross_val_score(
    best_ridge_model,
    X_train_scaled,
    y_train,
    cv = cross_val,
    scoring = 'r2'
)

print(best_r2_scores)


/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 

Optimal lambda: 0.1
Mean R^2: 0.7164037228590681
[0.71751345 0.73436974 0.76204491 0.76658304 0.75487552 0.7685196
 0.79112719 0.44783017 0.68993342 0.73124019]


/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 

**c)** Repeat this process for a different set of 10-folds, using a random state of your choosing.  Determine which $\lambda$ results in the optimal mean $R^2$, similar to what you did above.

In [19]:
cross_val_2 = KFold(n_splits = 10, shuffle = True, random_state = 12345)

best_alpha_2 = None
best_mean_r2_2 = -999

for m in range(1, 21):
    alph = m / 20

    ridge_model = Ridge(alpha = alph)
    r2_scores = cross_val_score(
        ridge_model,
        X_train_scaled,
        y_train,
        cv = cross_val_2,
        scoring = 'r2'
    )

    mean_r2 = r2_scores.mean()

    print("lambda =", alph)
    print("mean R^2 =", mean_r2)
    print()

    if mean_r2 > best_mean_r2_2:
        best_mean_r2_2 = mean_r2
        best_alpha_2 = alph

print("Optimal lambda:", best_alpha_2)
print("Optimal mean R^2:", best_mean_r2_2)


/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 

lambda = 0.05
mean R^2 = 0.7149201439580527

lambda = 0.1
mean R^2 = 0.7148592146028434

lambda = 0.15
mean R^2 = 0.7147738476307648

lambda = 0.2
mean R^2 = 0.7146692199509797

lambda = 0.25
mean R^2 = 0.714549512057279

lambda = 0.3
mean R^2 = 0.7144181113836696

lambda = 0.35
mean R^2 = 0.7142777706124548

lambda = 0.4
mean R^2 = 0.7141307317639778

lambda = 0.45
mean R^2 = 0.7139788240882401

lambda = 0.5
mean R^2 = 0.7138235417505328

lambda = 0.55
mean R^2 = 0.713666105824909

lambda = 0.6
mean R^2 = 0.7135075140219899

lambda = 0.65
mean R^2 = 0.7133485807710825

lambda = 0.7
mean R^2 = 0.7131899696736065

lambda = 0.75
mean R^2 = 0.7130322198906105

lambda = 0.8
mean R^2 = 0.7128757676825954

lambda = 0.85
mean R^2 = 0.7127209640567219

lambda = 0.9
mean R^2 = 0.7125680892742458

lambda = 0.95
mean R^2 = 0.7124173648146674

lambda = 1.0
mean R^2 = 0.7122689632715076

Optimal lambda: 0.05
Optimal mean R^2: 0.7149201439580527


/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 

**d)** I'd suggest turning to Q3 on Gradescope here.

I don't anticipate a need to perform any more calculations or analyses for this problem.  This space is available for any optional calculations or analyses that you'd like, although there are no points associated with this part.

## 4. Evaluating Our Best Model [1 point]

**a)** Refit the ridge regression model with the optimal $\lambda$ found in Question **3b**, but this time fit the model to the full training data.  Print the resulting coefficients.

In [22]:
final_ridge_model = Ridge(alpha = 0.1)

final_ridge_model.fit(X_train_scaled, y_train)

pd.Series(final_ridge_model.coef_, index = X_train_scaled.columns)


/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewmandhyan/Desktop/Stat207/mdm16/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


MEI         0.059524
CO2         0.073363
CH4         0.006150
N2O        -0.071037
CFC-11     -0.131523
CFC-12      0.211369
TSI         0.037075
Aerosols   -0.046126
dtype: float64

**b)** You'll be asked to calculate the $R^2$ on the test data for the model from part **a** above.  

You will need to perform additional calculations or analyses to do this.  This space is available for those calculations, although there are no points associated with the code.  You will enter the results of your calculation on **Q4.2** on Gradescope.

In [23]:
test_r2 = final_ridge_model.score(X_test_scaled, y_test)

test_r2


0.1691085027102034

## 5. AI Acknowledgement

Our course policy is that you should write all of your own interpretations and other narrative answers (phrases or sentences) yourself without the assistance of AI.  You may use AI to help guide your code, although you should write all of your own code yourself (not copy-paste from another source) and you should cite your use of AI.  I would encourage you to try to generate any necessary code yourself first using course resources and using AI as a debugging tool if/when you reach an error that you can't figure out or to help you perform any coding tasks that are more advanced than we've demonstrated during class (intended only for projects).  

Did you use AI on this assignment?  Did you use other resources outside of our course-provided resources on this assignment?

no

If you used AI or other resources, answer the following questions to cite your usage.

- Which AI and/or resources did you use (including links, if appropriate)?
- What prompts did you ask it?
- How did you integrate the responses into your assignment?  Specifically, which questions or parts are associated with this usage?

Note: answering these three questions are enough for our course but may not be enough for a different course or context.

n/a

Remember to keep all your cells and hit the save icon above periodically to checkpoint (save) your results on your local computer. Once you are satisified with your results restart the kernel and run all (Kernel -> Restart & Run All). **Make sure nothing has changed**. Checkpoint and exit (File -> Save and Checkpoint + File -> Close and Halt). Follow the instructions on the Homework 11 Canvas Assignment to submit your notebook to GitHub.